In [3]:
! pip install yfinance python-dotenv 

In [4]:
import pandas as pd 
import numpy as np
import yfinance as yf 
from dotenv import load_dotenv
import os 
load_dotenv()

False

In [5]:
TICKERS_UK = ["AZN.L", "GSK.L"]
TICKERS_USA = ["PFE", "JNJ", "MRK"]
BENCHMARK = "XLV"
ALL_TICKERS = TICKERS_UK + TICKERS_USA + [BENCHMARK]
start_date = "2025-01-01"
end_date = "2026-07-28"
rsi_window = 14
sma_window = 20
vol_window = 20 
risk_free_returns = 0.045

In [6]:
def fetch_ticker_sample(ticker: str, start: str, end:str, multi_level_index: bool= False ):
  df = yf.download(ticker, start=start, end=end, multi_level_index= False )
  if df.empty:
    raise ValueError(f"no data found for {ticker}")   
  df = df.reset_index()
  df.columns = df.columns.str.lower().str.strip()  
  df["ticker"]= ticker
  df["currency"]= "GBP" if ticker in TICKERS_UK else "USD"
  return df       

In [7]:
df = fetch_ticker_sample(ticker= "AZN.L", start= start_date, end= end_date )

[*********************100%***********************]  1 of 1 completed


In [8]:
df

,date,close,high,low,open,volume,ticker,currency
0,2025-01-02,10626.646484,10646.640175,10310.746175,10402.717151,2588002,AZN.L,GBP
1,2025-01-03,10590.658203,10782.597638,10524.679022,10652.638646,2879413,AZN.L,GBP
2,2025-01-06,10708.620117,10726.614438,10548.670601,10548.670601,2451616,AZN.L,GBP
3,2025-01-07,10664.633789,10710.619274,10522.678596,10594.655877,2763990,AZN.L,GBP
4,2025-01-08,10750.606445,10764.602027,10492.687860,10660.634846,1222394,AZN.L,GBP
...,...,...,...,...,...,...,...,...
375,2026-06-29,14298.000000,14388.000000,14200.000000,14364.000000,1883887,AZN.L,GBP
376,2026-06-30,14100.000000,14536.000000,14054.000000,14356.000000,2513351,AZN.L,GBP
377,2026-07-01,13854.000000,14232.000000,13838.000000,14138.000000,1804499,AZN.L,GBP
378,2026-07-02,14538.000000,14686.000000,13894.000000,13916.000000,2810573,AZN.L,GBP


In [9]:
frames = []
failed = []
for t in ALL_TICKERS:
  try:
    frames.append(fetch_ticker_sample(t, start_date, end_date))
  except:
    failed.append((t, str(e)))
if failed:
    print("Failed downloads:")
    for t, err in failed:
        print(f"  {t}: {err}")    

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [10]:
df =pd.concat(frames, ignore_index = True )
df.tail()

,date,close,high,low,open,volume,ticker,currency
2255,2026-06-26,160.339996,160.639999,156.179993,156.320007,16563800,XLV,USD
2256,2026-06-29,160.740005,161.089996,159.880005,160.699997,12935000,XLV,USD
2257,2026-06-30,158.660004,161.250000,157.960007,161.009995,11559400,XLV,USD
2258,2026-07-01,159.539993,159.960007,158.500000,159.610001,10123700,XLV,USD
2259,2026-07-02,163.740005,163.850006,160.380005,160.669998,14719400,XLV,USD


In [11]:
quality_report = (
    df.groupby("ticker")
    .agg(rows=("date", "count"),
         nulls_close=("close", lambda s: s.isna().sum()),
         duplicate_dates=("date", lambda s: s.duplicated().sum()))
)
quality_report

,rows,nulls_close,duplicate_dates
ticker,,,
AZN.L,380,0,0
GSK.L,380,0,0
JNJ,375,0,0
MRK,375,0,0
PFE,375,0,0
XLV,375,0,0


In [12]:
def engineer_features(g: pd.DataFrame) -> pd.DataFrame:
    g = g.sort_values("date").copy()

    g["daily_return"] = g["close"].pct_change() * 100

    g["sma_20"] = g["close"].rolling(sma_window).mean()
    rolling_std = g["close"].rolling(sma_window).std()
    g["bollinger_upper"] = g["sma_20"] + 2 * rolling_std
    g["bollinger_lower"] = g["sma_20"] - 2 * rolling_std

    change = g["close"].diff()
    gain = change.clip(lower=0)
    loss = (-change).clip(lower=0)
    avg_gain = gain.ewm(alpha=1 / rsi_window, min_periods=rsi_window, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / rsi_window, min_periods=rsi_window, adjust=False).mean()
    avg_loss_protected = avg_loss.replace(0, 1e-5)
    rs = avg_gain / avg_loss_protected
    g["rsi_14"] = (100 - (100 / (1 + rs))).clip(0, 100)

    g["volatility_20d"] = g["daily_return"].rolling(vol_window).std() * np.sqrt(252)

    running_max = g["close"].cummax()
    g["drawdown_pct"] = (g["close"] - running_max) / running_max * 100

    return g

featured = df.groupby("ticker", group_keys=False).apply(engineer_features)
featured.tail(10)

C:\Users\sai kiran\AppData\Local\Temp\ipykernel_14592\719196550.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  featured = df.groupby("ticker", group_keys=False).apply(engineer_features)


,date,close,high,low,open,volume,ticker,currency,daily_return,sma_20,bollinger_upper,bollinger_lower,rsi_14,volatility_20d,drawdown_pct
2250,2026-06-18,148.743988,150.605787,148.136666,150.605787,10879300,XLV,USD,-0.869227,150.161747,155.097552,145.225942,47.728204,18.141683,-6.358192
2251,2026-06-22,150.059998,150.619995,149.059998,149.330002,11706100,XLV,USD,0.884748,150.289773,155.065042,145.514503,51.635788,18.239625,-5.529698
2252,2026-06-23,152.179993,152.429993,151.000000,151.490005,9479900,XLV,USD,1.412765,150.437180,155.256753,145.617608,57.188010,18.445577,-4.195055
2253,2026-06-24,153.350006,154.669998,152.460007,153.130005,10070100,XLV,USD,0.768835,150.711785,155.538424,145.885146,59.922545,18.173404,-3.458472
2254,2026-06-25,155.630005,157.210007,153.589996,153.949997,9419200,XLV,USD,1.486794,151.086452,156.224714,145.948189,64.659692,18.748430,-2.023099
2255,2026-06-26,160.339996,160.639999,156.179993,156.320007,16563800,XLV,USD,3.026403,151.592576,158.164575,145.020576,72.017835,20.836461,0.000000
2256,2026-06-29,160.740005,161.089996,159.880005,160.699997,12935000,XLV,USD,0.249476,152.188892,159.783897,144.593886,72.540738,20.296757,0.000000
2257,2026-06-30,158.660004,161.250000,157.960007,161.009995,11559400,XLV,USD,-1.294016,152.762350,160.499032,145.025667,65.668825,20.505925,-1.294016
2258,2026-07-01,159.539993,159.960007,158.500000,159.610001,10123700,XLV,USD,0.554639,153.451491,161.014320,145.888662,67.089293,19.871589,-0.746555
2259,2026-07-02,163.740005,163.850006,160.380005,160.669998,14719400,XLV,USD,2.632576,154.293385,162.507192,146.079577,72.860890,21.302751,0.000000


In [13]:
return_wide = featured.pivot(index ="date", columns= "ticker", values="daily_return")
correlation_matrix = return_wide.corr()
correlation_matrix.round(2)

ticker,AZN.L,GSK.L,JNJ,MRK,PFE,XLV
ticker,,,,,,
AZN.L,1.00,0.62,0.27,0.34,0.30,0.34
GSK.L,0.62,1.00,0.32,0.36,0.36,0.35
JNJ,0.27,0.32,1.00,0.50,0.40,0.59
MRK,0.34,0.36,0.50,1.00,0.59,0.69
PFE,0.30,0.36,0.40,0.59,1.00,0.65
XLV,0.34,0.35,0.59,0.69,0.65,1.00


In [14]:
correlation_matrix

ticker,AZN.L,GSK.L,JNJ,MRK,PFE,XLV
ticker,,,,,,
AZN.L,1.000000,0.619002,0.273390,0.339014,0.301748,0.340842
GSK.L,0.619002,1.000000,0.320120,0.358112,0.360352,0.347657
JNJ,0.273390,0.320120,1.000000,0.498578,0.402523,0.587468
MRK,0.339014,0.358112,0.498578,1.000000,0.589766,0.687901
PFE,0.301748,0.360352,0.402523,0.589766,1.000000,0.645844
XLV,0.340842,0.347657,0.587468,0.687901,0.645844,1.000000


In [15]:
correlation_matrix.columns.name = None
correlation_matrix.round(2)

,AZN.L,GSK.L,JNJ,MRK,PFE,XLV
ticker,,,,,,
AZN.L,1.00,0.62,0.27,0.34,0.30,0.34
GSK.L,0.62,1.00,0.32,0.36,0.36,0.35
JNJ,0.27,0.32,1.00,0.50,0.40,0.59
MRK,0.34,0.36,0.50,1.00,0.59,0.69
PFE,0.30,0.36,0.40,0.59,1.00,0.65
XLV,0.34,0.35,0.59,0.69,0.65,1.00


In [16]:
correlation_matrix = correlation_matrix.reset_index()

In [17]:
correlation_matrix

,ticker,AZN.L,GSK.L,JNJ,MRK,PFE,XLV
0,AZN.L,1.000000,0.619002,0.273390,0.339014,0.301748,0.340842
1,GSK.L,0.619002,1.000000,0.320120,0.358112,0.360352,0.347657
2,JNJ,0.273390,0.320120,1.000000,0.498578,0.402523,0.587468
3,MRK,0.339014,0.358112,0.498578,1.000000,0.589766,0.687901
4,PFE,0.301748,0.360352,0.402523,0.589766,1.000000,0.645844
5,XLV,0.340842,0.347657,0.587468,0.687901,0.645844,1.000000


In [18]:
benchmark_returns = return_wide[BENCHMARK]

betas = {}
for t in TICKERS_UK + TICKERS_USA:
    paired = return_wide[[t, BENCHMARK]].dropna()
    cov = paired[t].cov(paired[BENCHMARK])
    var = paired[BENCHMARK].var()
    betas[t] = cov / var

beta_summary = pd.Series(betas, name="beta_vs_XLV").sort_values(ascending=False)
beta_summary

MRK      1.187848
PFE      0.934294
JNJ      0.677922
GSK.L    0.544640
AZN.L    0.525285
Name: beta_vs_XLV, dtype: float64

In [19]:
daily_rf = risk_free_returns / 252

sharpe = {}
for t in TICKERS_UK + TICKERS_USA:
    r = return_wide[t].dropna() / 100
    excess = r - daily_rf
    sharpe[t] = (excess.mean() / excess.std()) * np.sqrt(252)

sharpe_summary = pd.Series(sharpe, name="sharpe_ratio").sort_values(ascending=False)
sharpe_summary


JNJ      2.081620
GSK.L    0.953109
AZN.L    0.744311
MRK      0.725318
PFE     -0.025585
Name: sharpe_ratio, dtype: float64

In [20]:
def tag_signals(g: pd.DataFrame) -> pd.DataFrame:
    g = g.sort_values("date").copy()
    g["signal"] = np.select(
        [g["rsi_14"] < 30, g["rsi_14"] > 70],
        ["oversold", "overbought"],
        default="none"
    )
    g["fwd_return_5d"]  = g["close"].shift(-5)  / g["close"] * 100 - 100
    g["fwd_return_10d"] = g["close"].shift(-10) / g["close"] * 100 - 100
    return g

signals = featured.groupby("ticker", group_keys=False).apply(tag_signals)

C:\Users\sai kiran\AppData\Local\Temp\ipykernel_14592\104444332.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  signals = featured.groupby("ticker", group_keys=False).apply(tag_signals)


In [21]:
signal_days = signals[signals["signal"] != "none"].dropna(subset=["fwd_return_5d", "fwd_return_10d"])

backtest_summary = (
    signal_days.groupby("signal")
    .agg(n_occurrences=("signal", "count"),
         avg_fwd_return_5d=("fwd_return_5d", "mean"),
         win_rate_5d=("fwd_return_5d", lambda s: (s > 0).mean() * 100),
         avg_fwd_return_10d=("fwd_return_10d", "mean"),
         win_rate_10d=("fwd_return_10d", lambda s: (s > 0).mean() * 100))
)
backtest_summary.round(2)

,n_occurrences,avg_fwd_return_5d,win_rate_5d,avg_fwd_return_10d,win_rate_10d
signal,,,,,
overbought,211,0.28,53.55,0.36,54.03
oversold,46,0.89,58.70,3.12,73.91


In [22]:
backtest_summary = backtest_summary.reset_index()

In [24]:
backtest_summary

,signal,n_occurrences,avg_fwd_return_5d,win_rate_5d,avg_fwd_return_10d,win_rate_10d
0,overbought,211,0.276062,53.554502,0.357468,54.028436
1,oversold,46,0.886400,58.695652,3.115287,73.913043


In [25]:
per_ticker_backtest = (
    signal_days.groupby(["ticker", "signal"])
    .agg(n=("signal", "count"), avg_fwd_return_5d=("fwd_return_5d", "mean"))
    .round(2)
)
per_ticker_backtest

n  avg_fwd_return_5d
ticker signal                           
AZN.L  overbought  31              -0.27
       oversold     2               4.35
GSK.L  overbought  25              -0.01
       oversold    13               1.56
JNJ    overbought  92               0.76
       oversold     2               0.64
MRK    overbought  27               1.20
       oversold    15               0.29
PFE    overbought   7              -4.60
       oversold     6              -0.50
XLV    overbought  29              -0.12
       oversold     8               1.16

In [26]:
per_ticker_backtest = per_ticker_backtest.reset_index()
per_ticker_backtest

,ticker,signal,n,avg_fwd_return_5d
0,AZN.L,overbought,31,-0.27
1,AZN.L,oversold,2,4.35
2,GSK.L,overbought,25,-0.01
3,GSK.L,oversold,13,1.56
4,JNJ,overbought,92,0.76
5,JNJ,oversold,2,0.64
6,MRK,overbought,27,1.20
7,MRK,oversold,15,0.29
8,PFE,overbought,7,-4.60
9,PFE,oversold,6,-0.50


In [27]:
fact_export = featured[[
    "date", "ticker", "currency", "open", "high", "low", "close", "volume",
    "daily_return", "sma_20", "bollinger_upper", "bollinger_lower",
    "rsi_14", "volatility_20d", "drawdown_pct"
]].copy()
fact_export.to_csv("pharma_fact_indicators.csv", index=False)

correlation_matrix.to_csv("pharma_correlation_matrix.csv")
beta_summary.to_frame().to_csv("pharma_beta_vs_benchmark.csv")
sharpe_summary.to_frame().to_csv("pharma_sharpe_ratios.csv")
backtest_summary.to_csv("pharma_signal_backtest_summary.csv")

print("Exported: pharma_fact_indicators.csv, pharma_correlation_matrix.csv,",
      "pharma_beta_vs_benchmark.csv, pharma_sharpe_ratios.csv,",
      "pharma_signal_backtest_summary.csv")

Exported: pharma_fact_indicators.csv, pharma_correlation_matrix.csv, pharma_beta_vs_benchmark.csv, pharma_sharpe_ratios.csv, pharma_signal_backtest_summary.csv
